# Final Evaluation: 7-Day Average

This notebook will recompute the 7-day-average MAE using the same three 28-day validation windows. We’ll use the results to resolve the earlier window 2 score mismatch.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "code" / "notebooks").is_dir()
    ),
    Path.cwd(),
)
data_dir = project_dir / "data" / "raw" / "m5-forecasting-accuracy"

all_sales = pd.read_csv(data_dir / "sales_train_validation.csv")
daily_sales_all = all_sales.iloc[:, 6:]

print("Full table:", all_sales.shape)
print("Daily sales:", daily_sales_all.shape)

In [ ]:
train_sales_window_1 = daily_sales_all.iloc[:, :-56]
validation_sales_window_1 = daily_sales_all.iloc[:, -56:-28]

train_sales_window_2 = daily_sales_all.iloc[:, :-84]
validation_sales_window_2 = daily_sales_all.iloc[:, -84:-56]

train_sales_latest = daily_sales_all.iloc[:, :-28]
validation_sales_latest = daily_sales_all.iloc[:, -28:]

for name, train_data, validation_data in [
    ("Window 1", train_sales_window_1, validation_sales_window_1),
    ("Window 2", train_sales_window_2, validation_sales_window_2),
    ("Latest", train_sales_latest, validation_sales_latest),
]:
    print(
        name,
        "| train:", train_data.shape,
        train_data.columns[[0, -1]].tolist(),
        "| validation:", validation_data.shape,
        validation_data.columns[[0, -1]].tolist()
    )

In [ ]:
import numpy as np

def calculate_7day_mae(train_data, validation_data):
    average_last_7 = train_data.iloc[:, -7:].mean(axis=1).to_numpy()
    actual_sales = validation_data.to_numpy()

    # Broadcast each series' 7-day average across its 28 validation days
    return np.abs(actual_sales - average_last_7[:, None]).mean()


baseline_7day_recalculated = pd.DataFrame([
    {
        "window": "Window 1",
        "MAE": calculate_7day_mae(
            train_sales_window_1, validation_sales_window_1
        )
    },
    {
        "window": "Window 2",
        "MAE": calculate_7day_mae(
            train_sales_window_2, validation_sales_window_2
        )
    },
    {
        "window": "Latest",
        "MAE": calculate_7day_mae(
            train_sales_latest, validation_sales_latest
        )
    }
])

baseline_7day_recalculated

## Recalculated 7-day-average results

These scores cover all 30,490 series. We’ll use the 7-day average as the project’s working forecasting method because it is simple and light on memory. CatBoost’s pilot had a slightly lower mean MAE, but its results varied across windows and it required more resources.

In [ ]:
baseline_7day_final = baseline_7day_recalculated.copy()

baseline_7day_final.loc[len(baseline_7day_final)] = [
    "Mean",
    baseline_7day_final["MAE"].mean()
]

baseline_7day_final["MAE"] = baseline_7day_final["MAE"].round(6)

baseline_7day_final

## Create a 28-day forecast for inventory planning

For each of the 30,490 product-store series, we’ll average sales over the latest seven observed days, d_1907–d_1913, then use that average as the forecast for each of the next 28 days, d_1914–d_1941. This is the forward forecast that we’ll use in the inventory-planning phase.

In [ ]:
last_7_day_average_all = daily_sales_all.iloc[:, -7:].mean(axis=1)

future_forecast_days = [
    f"d_{day}" for day in range(1914, 1942)
]

forecast_28day_all = pd.DataFrame(
    np.repeat(
        last_7_day_average_all.to_numpy()[:, None],
        len(future_forecast_days),
        axis=1
    ),
    index=all_sales["id"].to_numpy(),
    columns=future_forecast_days
)

forecast_28day_all.shape, forecast_28day_all.iloc[:5, :5]

In [ ]:
forecast_with_metadata = (
    all_sales[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]]
    .set_index("id")
    .join(forecast_28day_all)
)

forecast_with_metadata["forecast_28day_total"] = (
    forecast_with_metadata[future_forecast_days].sum(axis=1)
)

forecast_with_metadata[
    [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "forecast_28day_total"
    ]
].head()

In [ ]:
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

forecast_file = processed_dir / "forecast_7day_average_28day.csv"
forecast_with_metadata.to_csv(forecast_file, index=True, index_label="id")

forecast_file